# Basic EDA - Time Series

In [ ]:
# Import libraries for data manipulation, stats, and plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## 1. Load & Inspect Data

In [ ]:
# Load CSV, parse dates, set daily frequency
df = pd.read_csv('data/timeseries.csv', parse_dates=['date'], index_col='date')
df = df.asfreq('D')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Summary statistics (count, mean, min, quartiles, max)
df.describe()

In [ ]:
# Visualise the full series to spot trends and patterns
df.plot(title='Daily Unit Sales')
plt.show()

## 2. Handle Outliers & Missing Values

In [ ]:
# Cap outliers via IQR, forward/backward fill missing values
Q1, Q3 = df['unit_sales'].quantile([0.25, 0.75])
lower, upper = Q1 - 1.5 * (Q3 - Q1), Q3 + 1.5 * (Q3 - Q1)
print(f'Outliers: {((df["unit_sales"] < lower) | (df["unit_sales"] > upper)).sum()}')

df['unit_sales'] = df['unit_sales'].clip(lower, upper).ffill().bfill()
print(f'Missing after fill: {df.isnull().sum().values[0]}')

## 3. Seasonal Decomposition

In [ ]:
# Decompose into trend, seasonal (7-day), and residual components
fig = seasonal_decompose(df['unit_sales'], model='additive', period=7).plot()
fig.set_size_inches(12, 8)
plt.show()

## 4. Stationarity (ADF Test)

In [ ]:
# Augmented Dickey-Fuller test for stationarity
result = adfuller(df['unit_sales'].dropna())
print(f'ADF: {result[0]:.4f}, p-value: {result[1]:.4f}')
print('Stationary' if result[1] < 0.05 else 'Non-stationary')

## 5. ACF & PACF

**ACF (Autocorrelation Function):** shows correlation between the series and its lagged values (including indirect effects propagated through intermediate lags)

**PACF (Partial Autocorrelation Function):** shows the direct correlation at each lag, controlling for shorter lags.

In [ ]:
# Autocorrelation and partial autocorrelation (up to 30 lags)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(df['unit_sales'], lags=30, ax=ax1)
plot_pacf(df['unit_sales'], lags=30, ax=ax2)
plt.show()

Significant bars beyond the blue confidence band indicate important lags

## 6. Save Cleaned Data

In [ ]:
# Export cleaned dataset for downstream models
df.to_csv('data/cleaned_timeseries.csv')
print('Saved to data/cleaned_timeseries.csv')